In [2]:
# ============================================================
# 8. LEAKAGE-SAFE TRAIN / TEST PREPARATION
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

# ============================================================
# 8.1 FINAL ML FEATURE SET
# ============================================================

# Only ORIGINAL behavioral measurements
# Do NOT include normalization or BPI-derived scores.

BEHAVIORAL_FEATURES = [
    "mean_velocity",
    "std_velocity",
    "median_velocity",
    "max_velocity",
    "min_velocity",

    "mean_acceleration",
    "std_acceleration",
    "max_acceleration",

    "mean_direction_change",
    "median_direction_change",

    "total_distance",
    "mean_distance",
    "window_duration",

    "idle_events",
    "idle_time",
    "idle_percentage",

    "pause_count",
    "mean_pause",
    "median_pause",
    "std_pause",
    "pause_95",

    "velocity_entropy",
    "direction_entropy",
    "velocity_cv",
    "smoothness",

    "num_events"
]

X = df[BEHAVIORAL_FEATURES].copy()
y = df[TARGET].copy()

print("Number of behavioral features:", len(BEHAVIORAL_FEATURES))
print("\nFeatures:")
print(BEHAVIORAL_FEATURES)

print("\nX shape:", X.shape)
print("y shape:", y.shape)


# ------------------------------------------------------------
# Handle invalid values
# ------------------------------------------------------------

X = X.replace([np.inf, -np.inf], np.nan)

print("\nMissing values:")
print(X.isna().sum()[X.isna().sum() > 0])


# Median imputation
X = X.fillna(X.median())

print("\nMissing values after imputation:")
print(X.isna().sum().sum())


# ------------------------------------------------------------
# Recreate the SAME leakage-safe session split
# ------------------------------------------------------------

groups = (
    df["user"].astype(str)
    + "_"
    + df["session"].astype(str)
)

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]


# ------------------------------------------------------------
# Verify no session leakage
# ------------------------------------------------------------

train_sessions = set(groups_train)
test_sessions = set(groups_test)

overlap = train_sessions.intersection(test_sessions)

print("\nSession leakage check:")
print("Train sessions:", len(train_sessions))
print("Test sessions:", len(test_sessions))
print("Overlapping sessions:", len(overlap))

assert len(overlap) == 0

print("\nFinal ML dataset:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


# ------------------------------------------------------------
# 8.2 Check target
# ------------------------------------------------------------

TARGET = "productivity_score"

print("\nTarget statistics:")
print(df[TARGET].describe())


# ------------------------------------------------------------
# 8.3 Define columns that must NOT enter the model
# ------------------------------------------------------------

ID_COLUMNS = [
    "user",
    "session",
    "window_id"
]

# Keep only ID columns that actually exist
ID_COLUMNS = [col for col in ID_COLUMNS if col in df.columns]

print("\nID columns excluded from model:")
print(ID_COLUMNS)


# ------------------------------------------------------------
# 8.4 Build X and y
# ------------------------------------------------------------

# Numeric behavioral features only
feature_columns = [
    col for col in df.columns
    if col not in ID_COLUMNS + [TARGET]
    and pd.api.types.is_numeric_dtype(df[col])
]

X = df[feature_columns].copy()
y = df[TARGET].copy()

print("\nNumber of features:", len(feature_columns))
print("Features:")
print(feature_columns)

print("\nX shape:", X.shape)
print("y shape:", y.shape)


# ------------------------------------------------------------
# 8.5 Handle invalid values
# ------------------------------------------------------------

# Replace infinite values
X = X.replace([np.inf, -np.inf], np.nan)

# Check missing values
missing = X.isna().sum()

print("\nMissing values:")
print(missing[missing > 0])

# Median imputation for any missing feature values
X = X.fillna(X.median())


# ------------------------------------------------------------
# 8.6 Create session-level groups
# ------------------------------------------------------------

# A session belongs to exactly one user.
# Therefore (user, session) is our leakage-prevention group.

groups = (
    df["user"].astype(str)
    + "_"
    + df["session"].astype(str)
)

print("\nNumber of unique sessions:", groups.nunique())


# ------------------------------------------------------------
# 8.7 Leakage-safe train/test split
# ------------------------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]


# ------------------------------------------------------------
# 8.8 Verify there is NO session overlap
# ------------------------------------------------------------

train_sessions = set(groups_train)
test_sessions = set(groups_test)

overlap = train_sessions.intersection(test_sessions)

print("\nSession leakage check:")
print("Train sessions:", len(train_sessions))
print("Test sessions:", len(test_sessions))
print("Overlapping sessions:", len(overlap))

assert len(overlap) == 0, "LEAKAGE DETECTED!"


# ------------------------------------------------------------
# 8.9 Final dataset sizes
# ------------------------------------------------------------

print("\nFinal split:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


# ------------------------------------------------------------
# 8.10 Target distribution
# ------------------------------------------------------------

print("\nTraining target:")
print(y_train.describe())

print("\nTesting target:")
print(y_test.describe())

Number of behavioral features: 26

Features:
['mean_velocity', 'std_velocity', 'median_velocity', 'max_velocity', 'min_velocity', 'mean_acceleration', 'std_acceleration', 'max_acceleration', 'mean_direction_change', 'median_direction_change', 'total_distance', 'mean_distance', 'window_duration', 'idle_events', 'idle_time', 'idle_percentage', 'pause_count', 'mean_pause', 'median_pause', 'std_pause', 'pause_95', 'velocity_entropy', 'direction_entropy', 'velocity_cv', 'smoothness', 'num_events']

X shape: (10000, 26)
y shape: (10000,)

Missing values:
std_pause    127
dtype: int64

Missing values after imputation:
0

Session leakage check:
Train sessions: 72
Test sessions: 19
Overlapping sessions: 0

Final ML dataset:
X_train: (7980, 26)
X_test : (2020, 26)
y_train: (7980,)
y_test : (2020,)

Target statistics:
count    10000.000000
mean        54.527090
std         24.021172
min          0.000000
25%         38.251228
50%         55.496162
75%         72.813151
max        100.000000
Name: